---
# <div align="center"><font color='red'>  </font></div>
# <div align="center"><font color='red'> COSC 2779/2972 | Deep Learning  </font></div>
## <div align="center"> <font color='red'> Week 5 Lectorial Example: **Vision Applications & CNN Architectures**</font></div>
---

# The transfer learning pipeline

The bulk of this notebook walks through the seven steps of a transfer learning workflow. Keep these in view — each numbered `Step N` heading below maps directly to one of them:

1. **Select a related proxy task** that has a large dataset (or a pre-trained model).
2. **Train the selected base model** on the proxy task — or download the pre-trained weights.
3. **Identify the feature layer** you want to use for your task (the *bottleneck*).
4. **Remove the layers above the bottleneck** and reconfigure them to your task — attach a **head**.
5. **Freeze all the layers** in the base model.
6. **Train the network** on the new task.
7. *(Optional)* **Unfreeze some top layers** in the base model and **fine-tune**.

Steps 1–2 are usually a one-liner in modern frameworks (`weights=…IMAGENET1K…`). The interesting mechanics are in steps 3–7.

# Setup Notebook

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Subset

from torchvision import transforms
from torchvision.datasets import OxfordIIITPet
from torchvision.models import (
    vgg16, VGG16_Weights,
    resnet50, ResNet50_Weights,
)

from sklearn.metrics import roc_auc_score

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Using device:', device)
torch.manual_seed(42)


In [ ]:
from itertools import cycle

def plotter(history_hold, metric='loss', ylim=None):
    cycol = cycle('bgrcmk')
    for name, hist in history_hold.items():
        y_train = hist[metric]
        y_val   = hist['val_' + metric]
        x = np.arange(len(y_val))
        c = next(cycol)
        plt.plot(x, y_train, c + '-',  label=name + '_train')
        plt.plot(x, y_val,   c + '--', label=name + '_val')
    plt.legend()
    plt.xlim([1, max(plt.xlim())])
    if ylim is not None:
        plt.ylim(ylim)
    plt.xlabel('Epoch')
    plt.ylabel(metric)
    plt.grid(True)

m_histories = {}


# Using a torchvision pre-built model

Before we build a transfer learning pipeline, let's look at one of the classic pre-built CNN architectures shipped in `torchvision.models`.

Loading VGG16 with ImageNet weights is a single call — this is essentially **steps 1 and 2** of the transfer learning pipeline done for you.

In [ ]:
VGG16_model = vgg16(weights=VGG16_Weights.IMAGENET1K_V1)
print(VGG16_model)


A textual summary (parameter counts + shapes) via `torchinfo`:

In [ ]:
# Install once per environment
%pip install --quiet torchinfo


In [ ]:
from torchinfo import summary
summary(VGG16_model, input_size=(1, 3, 224, 224), col_names=('input_size', 'output_size', 'num_params'))


Two things worth noting for what comes next:

- The network is split into two `Sequential` blocks: `.features` (all the convolutions and pools) and `.classifier` (the fully-connected head that produces the 1000 ImageNet logits).
- This `features` / `classifier` split is exactly where a transfer learning workflow will cut — `.features` is the base model we want to reuse, `.classifier` is what we throw away and replace.

# Example: Cats vs Dogs

## Dataset & Preprocessing

We use **Oxford-IIIT Pet dataset**, which `torchvision` exposes with a `binary-category` target — 0 = cat, 1 = dog. It's ~7.4k images.

First run downloads ~800 MB into `./data`; subsequent runs are cached.

In [ ]:
IMG_SIZE = 150
BATCH_SIZE = 32

# Raw transform for visualisation only (no normalisation yet)
raw_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
])

full_trainval = OxfordIIITPet(
    root='./data', split='trainval', target_types='binary-category',
    download=True, transform=raw_transform,
)
test_ds_raw = OxfordIIITPet(
    root='./data', split='test', target_types='binary-category',
    download=True, transform=raw_transform,
)

# 80/20 split of trainval into train / val
g = torch.Generator().manual_seed(42)
indices = torch.randperm(len(full_trainval), generator=g).tolist()
n_val = int(0.2 * len(indices))
train_idx, val_idx = indices[n_val:], indices[:n_val]

class_label = ['cat', 'dog']
print(f'Number of training samples:   {len(train_idx)}')
print(f'Number of validation samples: {len(val_idx)}')
print(f'Number of test samples:       {len(test_ds_raw)}')


Let's visualise some images.

In [ ]:
plt.figure(figsize=(10, 10))
for i in range(9):
    image, label = full_trainval[train_idx[i]]
    ax = plt.subplot(3, 3, i + 1)
    plt.imshow(image.permute(1, 2, 0).numpy())
    plt.title(class_label[int(label)])
    plt.axis('off')
plt.show()


### Standardizing the data

Our raw images have a variety of sizes. Each pixel is a 3-integer RGB value between 0 and 255. This isn't a great fit for feeding a neural network — we need to:

- Resize to a fixed 150×150.
- Normalise pixel values. The `torchvision` ImageNet-pretrained models expect inputs normalised with the ImageNet mean/std, so we bake that into the transform pipeline.

In PyTorch we typically apply normalisation in the `transforms.Compose` pipeline. This is idiomatic and works well with `DataLoader` workers.

In [ ]:
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]

# Training transform: augmentation + normalisation
train_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(degrees=10),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])
# Eval transform: just resize + normalise (no augmentation)
eval_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

train_full = OxfordIIITPet(root='./data', split='trainval', target_types='binary-category',
                            download=False, transform=train_transform)
val_full   = OxfordIIITPet(root='./data', split='trainval', target_types='binary-category',
                            download=False, transform=eval_transform)
test_ds    = OxfordIIITPet(root='./data', split='test',     target_types='binary-category',
                            download=False, transform=eval_transform)

train_ds = Subset(train_full, train_idx)
val_ds   = Subset(val_full,   val_idx)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,
                          num_workers=2, pin_memory=True)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, num_workers=2, pin_memory=True)
test_loader  = DataLoader(test_ds,  batch_size=BATCH_SIZE, num_workers=2, pin_memory=True)


### Visualising augmentation

When you don't have a large image dataset, it's good practice to artificially introduce sample diversity by applying random yet realistic transformations to the training images — random horizontal flipping and small random rotations. This exposes the model to different aspects of the training data while slowing down overfitting.

Let's visualise what the first training image looks like after several random passes of the augmentation pipeline. Because normalisation shifts values away from [0, 1], we un-normalise before displaying.

In [ ]:
def denormalize(t):
    mean = torch.tensor(IMAGENET_MEAN).view(3, 1, 1)
    std  = torch.tensor(IMAGENET_STD).view(3, 1, 1)
    return (t.cpu() * std + mean).clamp(0, 1).permute(1, 2, 0).numpy()

plt.figure(figsize=(10, 10))
for i in range(9):
    image, label = train_ds[0]           # augmentation is re-sampled every access
    ax = plt.subplot(3, 3, i + 1)
    plt.imshow(denormalize(image))
    plt.title(class_label[int(label)])
    plt.axis('off')
plt.show()


## Build a transfer learning model

Now we walk through the seven steps of the transfer learning pipeline explicitly.

### Step 1 — Select a related proxy task

For image classification, the canonical proxy task is **ImageNet** — 1.28M images, 1000 categories, huge visual variety. Features learned on ImageNet transfer well to almost any natural-image task.

### Step 2 — Train / download the base model

Our choice is **ResNet50**.

In [ ]:
base_model = resnet50(weights=ResNet50_Weights.IMAGENET1K_V2)
print(base_model)


### Step 3 — Identify the feature layer (the "bottleneck")

In `torchvision`'s ResNet50 the layout is:

```
conv1 -> bn1 -> relu -> maxpool
  -> layer1 -> layer2 -> layer3 -> layer4       (the convolutional backbone)
  -> avgpool -> fc                              (the classifier head)
```

The natural bottleneck is the output of `avgpool` — a 2048-dim vector per image. `fc` is the layer that maps those 2048 features to 1000 ImageNet classes. That's exactly the layer we don't want.

### Step 4 — Remove the top, attach a new head

For our binary cat vs. dog task, we replace `base_model.fc` with a small head:

- `Dropout(0.2)` for regularisation.
- `Linear(2048, 1)` for a single logit (binary classification with `BCEWithLogitsLoss`).

This mirrors the original Keras structure exactly (GAP → Dropout → Dense(1)).

In [ ]:
# Bottleneck feature size for ResNet50 is 2048.
bottleneck_dim = base_model.fc.in_features
print(f'Bottleneck feature dimensionality: {bottleneck_dim}')

# Step 4: replace the 1000-way classifier with our task-specific head.
base_model.fc = nn.Sequential(
    nn.Dropout(0.2),
    nn.Linear(bottleneck_dim, 1),      # single logit -> BCEWithLogitsLoss
)


### Step 5 — Freeze all layers in the base model

We freeze everything *except* the newly-created head. In PyTorch this means setting `requires_grad = False` on every parameter of the pre-existing layers, then leaving the new `fc` head trainable.

In [ ]:
# Freeze everything first...
for p in base_model.parameters():
    p.requires_grad = False
# ...then re-enable gradient for the new head.
for p in base_model.fc.parameters():
    p.requires_grad = True

trainable = sum(p.numel() for p in base_model.parameters() if p.requires_grad)
total     = sum(p.numel() for p in base_model.parameters())
print(f'Trainable / total parameters: {trainable:,} / {total:,}  (head-only)')


**One subtle PyTorch-specific point**

In PyTorch, batchnorm's train/eval behaviour is controlled by `model.train()` / `model.eval()`, not by `requires_grad`. Freezing weights alone doesn't freeze batchnorm running stats. If we wanted the strictest equivalent of the Keras behaviour after unfreezing, we'd call `.eval()` on the frozen batchnorm modules to keep them in inference mode. For this basic demo it doesn't matter much.

### Step 6 — Train the network on the new task

In [ ]:
def run_epoch(model, loader, criterion, optimizer=None):
    training = optimizer is not None
    model.train(training)
    total_loss, ys, preds, probs = 0.0, [], [], []
    ctx = torch.enable_grad() if training else torch.no_grad()
    with ctx:
        for xb, yb in loader:
            xb = xb.to(device)
            yb = yb.to(device).float()             # BCEWithLogitsLoss expects float
            logits = model(xb).squeeze(1)
            loss = criterion(logits, yb)
            if training:
                optimizer.zero_grad()
                loss.backward()
                optimizer.step()
            total_loss += loss.item() * len(xb)
            probs.extend(torch.sigmoid(logits).detach().cpu().numpy())
            preds.extend((logits > 0).long().detach().cpu().numpy())
            ys.extend(yb.long().cpu().numpy())
    n = len(ys)
    return {
        'loss':     total_loss / n,
        'accuracy': (np.array(preds) == np.array(ys)).mean(),
        'auc':      roc_auc_score(ys, probs) if len(set(ys)) > 1 else float('nan'),
    }


def train_and_track(model, train_loader, val_loader, epochs=5, lr=1e-3, verbose=True):
    model = model.to(device)
    optimizer = torch.optim.Adam(
        filter(lambda p: p.requires_grad, model.parameters()), lr=lr)
    criterion = nn.BCEWithLogitsLoss()

    hist = {k: [] for k in ['loss', 'accuracy', 'auc',
                             'val_loss', 'val_accuracy', 'val_auc']}
    for ep in range(epochs):
        tr = run_epoch(model, train_loader, criterion, optimizer)
        vl = run_epoch(model, val_loader,   criterion, optimizer=None)
        for k, v in tr.items(): hist[k].append(v)
        for k, v in vl.items(): hist['val_' + k].append(v)
        if verbose:
            print(f'ep {ep+1:2d}/{epochs}  '
                  f'loss {tr["loss"]:.3f}  acc {tr["accuracy"]:.3f}  |  '
                  f'val_loss {vl["loss"]:.3f}  val_acc {vl["accuracy"]:.3f}  val_auc {vl["auc"]:.3f}')
    return hist


In [ ]:
epochs = 20
m_histories['head_only'] = train_and_track(base_model, train_loader, val_loader, epochs=epochs, lr=1e-3)


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
plt.sca(axes[0]); plotter(m_histories, metric='loss')
plt.sca(axes[1]); plotter(m_histories, metric='accuracy', ylim=[0.0, 1.0])
plt.tight_layout(); plt.show()


### Step 7 — (Optional) Unfreeze top layers and fine-tune

The head is now warmed up on our task. We can squeeze more performance by *carefully* unfreezing the top layers of the backbone and continuing training at a much lower learning rate.

Two rules of thumb:

1. **Only fine-tune after the head has been trained.** Otherwise the large gradients coming from a randomly-initialised head will wreck the pretrained weights.
2. **Only unfreeze the top layers.** Lower layers learn generic edge/texture features; higher layers are more task-specific and benefit most from adaptation.

For ResNet50 the natural granularity is the four `layer1`…`layer4` stages. We'll unfreeze `layer4` (the last stage) and keep `layer1`…`layer3` frozen.

In [ ]:
# Unfreeze the last conv stage of the backbone.
for p in base_model.layer4.parameters():
    p.requires_grad = True

trainable = sum(p.numel() for p in base_model.parameters() if p.requires_grad)
total     = sum(p.numel() for p in base_model.parameters())
print(f'After unfreezing layer4  ->  trainable / total = {trainable:,} / {total:,}')


In [ ]:
# Fine-tune with a much smaller learning rate (10x smaller than head training).
m_histories['fine_tuned'] = train_and_track(
    base_model, train_loader, val_loader, epochs=5, lr=1e-4)

fig, axes = plt.subplots(1, 2, figsize=(14, 4))
plt.sca(axes[0]); plotter(m_histories, metric='loss')
plt.sca(axes[1]); plotter(m_histories, metric='accuracy', ylim=[0.0, 1.0])
plt.tight_layout(); plt.show()


### Wrap-up

The seven-step pipeline you just executed is the same regardless of task or architecture:

| Step | What we did |
|------|-------------|
| 1. Select proxy task            | ImageNet |
| 2. Get pretrained model         | `resnet50(weights=ResNet50_Weights.IMAGENET1K_V2)` |
| 3. Identify bottleneck          | Output of `avgpool`, 2048-dim |
| 4. Attach new head              | `nn.Sequential(Dropout(0.2), Linear(2048, 1))` |
| 5. Freeze base                  | `requires_grad = False` on backbone |
| 6. Train head                   | Adam, lr=1e-3, 20 epochs |
| 7. Fine-tune top of base        | Unfreeze `layer4`, Adam, lr=1e-4, 10 epochs |

Swap ResNet50 for MobileNetV2, EfficientNet, ViT, etc. and the workflow is identical — only the bottleneck name and dimensionality change.

## Test set evaluation

Only after all our development-set decisions have been made do we touch the test set.

In [ ]:
test_metrics = run_epoch(base_model, test_loader, nn.BCEWithLogitsLoss(), optimizer=None)
for k, v in test_metrics.items():
    print(f'test_{k:9s}: {v:.4f}')
